In [1]:


from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

In [2]:
# ============================================================
# 2. Cargar parquet desde Silver
# ============================================================

ruta_silver = Path("../data/Silver")

# Buscar archivos que empiecen por muestra_20_segovia
archivos = list(ruta_silver.glob("muestra_20_segovia*.parquet"))

print("Archivos encontrados:")
for archivo in archivos:
    print("-", archivo.name)

if len(archivos) == 0:
    raise FileNotFoundError("No se encontró ningún parquet que empiece por muestra_20_segovia en Silver")

ruta_parquet = archivos[0]

df_muestra = pd.read_parquet(ruta_parquet)

print("Ruta cargada:", ruta_parquet)
print("Shape:", df_muestra.shape)

display(df_muestra.head())

Archivos encontrados:
- muestra_20_segovia_18_madrid_2_restantes.parquet
Ruta cargada: ..\data\Silver\muestra_20_segovia_18_madrid_2_restantes.parquet
Shape: (20, 35)


,licitacion_id,titulo,detail_url,updated,expediente,tipo_contrato_codigo,lugar_ejecucion_codigo,cpv_codes,organo_contratacion,estado_codigo,...,url,cpv_descripcion,cpv_nivel,presentacion_hasta_dt,fecha_vigente,portal,dominio_url,cercania_segovia,orden_prioridad_geo,orden_no_madrid
0,4cbe6c463d894a88,Servicios sanitarios para la temporada de vera...,http://www.madrid.org/cs/Satellite?op2=PCON&id...,2018-02-23,C-336A/002-18 (A/SER-001984/2018),Servicios,ES300,85141200,"Consejería de Cultura, Turismo y Deportes",PUB,...,http://www.madrid.org/cs/Satellite?op2=PCON&id...,Servicios prestados por enfermeros,Detalle,2018-03-12,False,otro,www.madrid.org,madrid,3,NaN
1,075dac71258ec192,Servicio de análisis e informe de resultados d...,http://www.madrid.org/cs/Satellite?op2=PCON&id...,2018-03-01,GCASE1800003,Servicios,ES300,85145000,Hospital Universitario del Sureste,PUB,...,http://www.madrid.org/cs/Satellite?op2=PCON&id...,Servicios prestados por laboratorios médicos,Categoría,2018-03-14,False,otro,www.madrid.org,madrid,3,NaN
2,a765d556e7c58ca4,"Servicio de: extracción, traslado, destrucción...",http://www.madrid.org/cs/Satellite?op2=PCON&id...,2018-03-13,PA. SER-18/2018,Servicios,ES300,85140000,Servicio Madrileño de Salud,PUB,...,http://www.madrid.org/cs/Satellite?op2=PCON&id...,Servicios varios de salud,Clase,2018-03-26,False,otro,www.madrid.org,madrid,3,NaN
3,ef5caa38bddf3488,Servicio de laboratorio de análisis clínicos p...,http://www.madrid.org/cs/Satellite?op2=PCON&id...,2018-07-23,6011800138,Servicios,ES300,85145000,"Empresa Pública de Metro de Madrid, Sociedad A...",PUB,...,http://www.madrid.org/cs/Satellite?op2=PCON&id...,Servicios prestados por laboratorios médicos,Categoría,2018-08-14,False,otro,www.madrid.org,madrid,3,NaN
4,72cb60c324ca8f34,Servicio de laboratorio para la realización de...,http://www.madrid.org/cs/Satellite?op2=PCON&id...,2018-09-19,6011800195,Servicios,ES300,85145000,"Empresa Pública de Metro de Madrid, S.A.",PUB,...,http://www.madrid.org/cs/Satellite?op2=PCON&id...,Servicios prestados por laboratorios médicos,Categoría,2018-10-08,False,otro,www.madrid.org,madrid,3,NaN


In [4]:
df_muestra.columns

Index(['licitacion_id', 'titulo', 'detail_url', 'updated', 'expediente',
       'tipo_contrato_codigo', 'lugar_ejecucion_codigo', 'cpv_codes',
       'organo_contratacion', 'estado_codigo', 'fecha_publicacion',
       'procedimiento_codigo', 'importe_sin_impuestos', 'fuente_publicacion',
       'presentacion_hasta', 'presentacion_hora', 'notice_types',
       'organo_dir3', 'contrato_duracion', 'contrato_duracion_unidad',
       'ofertas_recibidas', 'adjudicatario', 'adjudicatario_nif', 'estado',
       'tipo_contrato', 'url', 'cpv_descripcion', 'cpv_nivel',
       'presentacion_hasta_dt', 'fecha_vigente', 'portal', 'dominio_url',
       'cercania_segovia', 'orden_prioridad_geo', 'orden_no_madrid'],
      dtype='object')

In [5]:
# ============================================================
# 1. Filtrar licitaciones de Madrid según cercania_segovia
# ============================================================

df_madrid = df_muestra[
    df_muestra["cercania_segovia"]
    .astype(str)
    .str.contains("madrid", case=False, na=False)
].copy()

print("Filas Madrid:", df_madrid.shape[0])

display(
    df_madrid[
        [
            "licitacion_id",
            "titulo",
            "cercania_segovia",
            "detail_url",
            "dominio_url"
        ]
    ]
)

Filas Madrid: 18


,licitacion_id,titulo,cercania_segovia,detail_url,dominio_url
0,4cbe6c463d894a88,Servicios sanitarios para la temporada de vera...,madrid,http://www.madrid.org/cs/Satellite?op2=PCON&id...,www.madrid.org
1,075dac71258ec192,Servicio de análisis e informe de resultados d...,madrid,http://www.madrid.org/cs/Satellite?op2=PCON&id...,www.madrid.org
2,a765d556e7c58ca4,"Servicio de: extracción, traslado, destrucción...",madrid,http://www.madrid.org/cs/Satellite?op2=PCON&id...,www.madrid.org
3,ef5caa38bddf3488,Servicio de laboratorio de análisis clínicos p...,madrid,http://www.madrid.org/cs/Satellite?op2=PCON&id...,www.madrid.org
4,72cb60c324ca8f34,Servicio de laboratorio para la realización de...,madrid,http://www.madrid.org/cs/Satellite?op2=PCON&id...,www.madrid.org
5,da1fec228406a862,Contratación de un servicio médico de neumolog...,madrid,http://www.madrid.org/cs/Satellite?op2=PCON&id...,www.madrid.org
6,2c2b7a36be594b70,Contratación de un servicio médico para la rea...,madrid,http://www.madrid.org/cs/Satellite?op2=PCON&id...,www.madrid.org
7,20851c82c510177f,Servicio médico en la especialidad de Cirugía ...,madrid,http://www.madrid.org/cs/Satellite?op2=PCON&id...,www.madrid.org
8,53bcbf153dc11a56,Servicio médico para la realización de consult...,madrid,http://www.madrid.org/cs/Satellite?op2=PCON&id...,www.madrid.org
9,f748bce950d2b1bb,Análisis para la obtención del estado de situa...,madrid,http://www.madrid.org/cs/Satellite?op2=PCON&id...,www.madrid.org


In [6]:
# ============================================================
# 2. Librerías para revisar las URLs
# ============================================================

import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [7]:
# ============================================================
# 3. Función para diagnosticar una URL
# ============================================================

def diagnosticar_url(url):
    """
    Revisa si una URL de licitación devuelve HTML, PDF u otro tipo.
    Además, si es HTML, busca enlaces internos a documentos PDF.
    """

    resultado = {
        "url": url,
        "status_code": None,
        "content_type": None,
        "es_html": False,
        "es_pdf_directo": False,
        "n_links_pdf": 0,
        "links_pdf": [],
        "n_tablas_html": 0,
        "error": None
    }

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0 Safari/537.36"
        )
    }

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=30,
            allow_redirects=True
        )

        resultado["status_code"] = response.status_code
        resultado["content_type"] = response.headers.get("content-type", "")

        content_type = resultado["content_type"].lower()

        if "application/pdf" in content_type or url.lower().endswith(".pdf"):
            resultado["es_pdf_directo"] = True
            return resultado

        if "text/html" in content_type or "<html" in response.text.lower():
            resultado["es_html"] = True

            soup = BeautifulSoup(response.text, "html.parser")

            tablas = soup.find_all("table")
            resultado["n_tablas_html"] = len(tablas)

            links_pdf = []

            for a in soup.find_all("a", href=True):
                href = a["href"]
                texto = a.get_text(" ", strip=True)

                href_completo = urljoin(url, href)

                texto_revision = f"{texto} {href_completo}".lower()

                if ".pdf" in texto_revision or "pdf" in texto_revision:
                    links_pdf.append(href_completo)

            links_pdf = list(dict.fromkeys(links_pdf))

            resultado["links_pdf"] = links_pdf
            resultado["n_links_pdf"] = len(links_pdf)

    except Exception as exc:
        resultado["error"] = str(exc)

    return resultado

In [8]:
# ============================================================
# 4. Diagnosticar todas las URLs de Madrid
# ============================================================

urls_madrid = df_madrid["detail_url"].dropna().unique().tolist()

print("URLs únicas Madrid:", len(urls_madrid))

diagnosticos = []

for i, url in enumerate(urls_madrid, start=1):
    print(f"Revisando {i}/{len(urls_madrid)}")
    diagnosticos.append(diagnosticar_url(url))

df_diag_madrid = pd.DataFrame(diagnosticos)

display(df_diag_madrid)

URLs únicas Madrid: 18
Revisando 1/18
Revisando 2/18
Revisando 3/18
Revisando 4/18
Revisando 5/18
Revisando 6/18
Revisando 7/18
Revisando 8/18
Revisando 9/18
Revisando 10/18
Revisando 11/18
Revisando 12/18
Revisando 13/18
Revisando 14/18
Revisando 15/18
Revisando 16/18
Revisando 17/18
Revisando 18/18


,url,status_code,content_type,es_html,es_pdf_directo,n_links_pdf,links_pdf,n_tablas_html,error
0,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,0,[],0,None
1,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,0,[],0,None
2,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,0,[],0,None
3,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,0,[],0,None
4,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,0,[],0,None
5,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,0,[],0,None
6,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,0,[],0,None
7,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,0,[],0,None
8,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,0,[],0,None
9,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,0,[],0,None


In [9]:
# ============================================================
# 5. Resumen del diagnóstico
# ============================================================

resumen_diag = pd.DataFrame({
    "metrica": [
        "total_urls",
        "urls_html",
        "urls_pdf_directo",
        "urls_con_links_pdf",
        "urls_con_error",
        "urls_con_tablas_html"
    ],
    "valor": [
        df_diag_madrid.shape[0],
        df_diag_madrid["es_html"].sum(),
        df_diag_madrid["es_pdf_directo"].sum(),
        (df_diag_madrid["n_links_pdf"] > 0).sum(),
        df_diag_madrid["error"].notna().sum(),
        (df_diag_madrid["n_tablas_html"] > 0).sum()
    ]
})

display(resumen_diag)

,metrica,valor
0,total_urls,18
1,urls_html,18
2,urls_pdf_directo,0
3,urls_con_links_pdf,0
4,urls_con_error,0
5,urls_con_tablas_html,0


In [10]:
# ============================================================
# 6. URLs que tienen PDFs o son PDF directo
# ============================================================

df_con_pdf = df_diag_madrid[
    (df_diag_madrid["es_pdf_directo"])
    | (df_diag_madrid["n_links_pdf"] > 0)
].copy()

print("URLs con PDF directo o enlaces PDF:", df_con_pdf.shape[0])

display(
    df_con_pdf[
        [
            "url",
            "es_pdf_directo",
            "n_links_pdf",
            "links_pdf"
        ]
    ]
)

URLs con PDF directo o enlaces PDF: 0


,url,es_pdf_directo,n_links_pdf,links_pdf


In [11]:
# ============================================================
# 7. Decisión: ¿procedemos solo con HTML?
# ============================================================

todas_html = df_diag_madrid["es_html"].all()
ninguna_pdf_directa = (df_diag_madrid["es_pdf_directo"].sum() == 0)
sin_links_pdf = (df_diag_madrid["n_links_pdf"].sum() == 0)
sin_errores = df_diag_madrid["error"].isna().all()

print("Todas son HTML:", todas_html)
print("Ninguna es PDF directo:", ninguna_pdf_directa)
print("No hay links PDF:", sin_links_pdf)
print("Sin errores:", sin_errores)

if todas_html and ninguna_pdf_directa and sin_links_pdf and sin_errores:
    print("Conclusión: se puede proceder con extracción HTML.")
else:
    print("Conclusión: no todo está en HTML limpio. Revisar PDFs, errores o links documentales antes de extraer.")

Todas son HTML: True
Ninguna es PDF directo: True
No hay links PDF: True
Sin errores: True
Conclusión: se puede proceder con extracción HTML.


In [ ]:
# ============================================================
# 1. Seleccionar una licitación de Madrid para inspección
# ============================================================

idx = 0

fila_prueba = df_madrid.iloc[idx]

licitacion_id_prueba = fila_prueba["licitacion_id"]
url_prueba = fila_prueba["detail_url"]

print("Licitación ID:", licitacion_id_prueba)
print("Título:", fila_prueba["titulo"])
print("URL:", url_prueba)

In [12]:
# ============================================================
# 1. Seleccionar una licitación de Madrid para inspección
# ============================================================

idx = 0

fila_prueba = df_madrid.iloc[idx]

licitacion_id_prueba = fila_prueba["licitacion_id"]
url_prueba = fila_prueba["detail_url"]

print("Licitación ID:", licitacion_id_prueba)
print("Título:", fila_prueba["titulo"])
print("URL:", url_prueba)

Licitación ID: 4cbe6c463d894a88
Título: Servicios sanitarios para la temporada de verano 2018, en las instalaciones de la Dirección General de Juventud y Deporte
URL: http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354697699879


In [13]:
# ============================================================
# 2. Descargar HTML de una licitación
# ============================================================

import requests
from bs4 import BeautifulSoup

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

response = requests.get(
    url_prueba,
    headers=headers,
    timeout=30,
    allow_redirects=True
)

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print("Tamaño HTML:", len(response.text))

html_prueba = response.text
soup_prueba = BeautifulSoup(html_prueba, "html.parser")

Status code: 404
Content-Type: text/html
Tamaño HTML: 1589


In [14]:
# ============================================================
# 1. Diagnóstico corregido de URLs Madrid
# ============================================================

def diagnosticar_url_v2(url):
    """
    Diagnóstico corregido:
    - HTML válido solo si status_code == 200
    - 404 no se considera página útil aunque venga como text/html
    """

    resultado = {
        "url": url,
        "status_code": None,
        "content_type": None,
        "es_html": False,
        "html_valido": False,
        "es_pdf_directo": False,
        "n_links_pdf": 0,
        "n_tablas_html": 0,
        "titulo_html": None,
        "error": None
    }

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=30,
            allow_redirects=True
        )

        resultado["status_code"] = response.status_code
        resultado["content_type"] = response.headers.get("content-type", "")

        content_type = resultado["content_type"].lower()
        html_text = response.text.lower()

        if "application/pdf" in content_type or url.lower().endswith(".pdf"):
            resultado["es_pdf_directo"] = True
            return resultado

        if "text/html" in content_type or "<html" in html_text:
            resultado["es_html"] = True

            soup = BeautifulSoup(response.text, "html.parser")

            resultado["titulo_html"] = (
                soup.title.get_text(" ", strip=True)
                if soup.title
                else None
            )

            resultado["n_tablas_html"] = len(soup.find_all("table"))

            links_pdf = []

            for a in soup.find_all("a", href=True):
                href = a["href"]
                texto = a.get_text(" ", strip=True)
                texto_revision = f"{texto} {href}".lower()

                if ".pdf" in texto_revision or "pdf" in texto_revision:
                    links_pdf.append(href)

            resultado["n_links_pdf"] = len(set(links_pdf))

        resultado["html_valido"] = (
            resultado["status_code"] == 200
            and resultado["es_html"]
        )

    except Exception as exc:
        resultado["error"] = str(exc)

    return resultado

In [15]:
# ============================================================
# 2. Ejecutar diagnóstico corregido
# ============================================================

diagnosticos_v2 = []

for i, url in enumerate(urls_madrid, start=1):
    print(f"Revisando {i}/{len(urls_madrid)}")
    diagnosticos_v2.append(diagnosticar_url_v2(url))

df_diag_madrid_v2 = pd.DataFrame(diagnosticos_v2)

display(df_diag_madrid_v2)

Revisando 1/18
Revisando 2/18
Revisando 3/18
Revisando 4/18
Revisando 5/18
Revisando 6/18
Revisando 7/18
Revisando 8/18
Revisando 9/18
Revisando 10/18
Revisando 11/18
Revisando 12/18
Revisando 13/18
Revisando 14/18
Revisando 15/18
Revisando 16/18
Revisando 17/18
Revisando 18/18


,url,status_code,content_type,es_html,html_valido,es_pdf_directo,n_links_pdf,n_tablas_html,titulo_html,error
0,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,False,0,0,Comunidad de Madrid - PÃ¡gina no disponible,None
1,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,False,0,0,Comunidad de Madrid - PÃ¡gina no disponible,None
2,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,False,0,0,Comunidad de Madrid - PÃ¡gina no disponible,None
3,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,False,0,0,Comunidad de Madrid - PÃ¡gina no disponible,None
4,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,False,0,0,Comunidad de Madrid - PÃ¡gina no disponible,None
5,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,False,0,0,Comunidad de Madrid - PÃ¡gina no disponible,None
6,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,False,0,0,Comunidad de Madrid - PÃ¡gina no disponible,None
7,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,False,0,0,Comunidad de Madrid - PÃ¡gina no disponible,None
8,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,False,0,0,Comunidad de Madrid - PÃ¡gina no disponible,None
9,http://www.madrid.org/cs/Satellite?op2=PCON&id...,404,text/html,True,False,False,0,0,Comunidad de Madrid - PÃ¡gina no disponible,None


In [16]:
# ============================================================
# 7. Ver URLs completas sin truncar
# ============================================================

pd.set_option("display.max_colwidth", None)

display(
    df_madrid[
        [
            "licitacion_id",
            "dominio_url",
            "url",
            "detail_url"
        ]
    ]
)

,licitacion_id,dominio_url,url,detail_url
0,4cbe6c463d894a88,www.madrid.org,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354697699879,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354697699879
1,075dac71258ec192,www.madrid.org,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354698822987,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354698822987
2,a765d556e7c58ca4,www.madrid.org,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354700731475,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354700731475
3,ef5caa38bddf3488,www.madrid.org,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354726371612,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354726371612
4,72cb60c324ca8f34,www.madrid.org,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734902275,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734902275
5,da1fec228406a862,www.madrid.org,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734993514,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734993514
6,2c2b7a36be594b70,www.madrid.org,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734654576,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734654576
7,20851c82c510177f,www.madrid.org,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734678972,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734678972
8,53bcbf153dc11a56,www.madrid.org,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354739034738,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_c

In [17]:
# ============================================================
# 8. Diagnóstico textual de las URLs
# ============================================================

df_check_urls = df_madrid[
    [
        "licitacion_id",
        "url",
        "detail_url"
    ]
].copy()

for col in ["url", "detail_url"]:
    df_check_urls[f"{col}_len"] = df_check_urls[col].astype(str).str.len()
    df_check_urls[f"{col}_tiene_espacios"] = df_check_urls[col].astype(str).str.contains(" ", na=False)
    df_check_urls[f"{col}_empieza_http"] = df_check_urls[col].astype(str).str.startswith("http")

display(df_check_urls)

,licitacion_id,url,detail_url,url_len,url_tiene_espacios,url_empieza_http,detail_url_len,detail_url_tiene_espacios,detail_url_empieza_http
0,4cbe6c463d894a88,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354697699879,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354697699879,206,False,True,206,False,True
1,075dac71258ec192,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354698822987,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354698822987,206,False,True,206,False,True
2,a765d556e7c58ca4,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354700731475,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354700731475,206,False,True,206,False,True
3,ef5caa38bddf3488,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354726371612,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354726371612,206,False,True,206,False,True
4,72cb60c324ca8f34,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734902275,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734902275,206,False,True,206,False,True
5,da1fec228406a862,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734993514,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734993514,206,False,True,206,False,True
6,2c2b7a36be594b70,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734654576,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734654576,206,False,True,206,False,True
7,20851c82c510177f,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734678972,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contratosPublicos&language=es&idConsejeria=1109266187224&cid=1354734678972,206,False,True,206,False,True
8,53bcbf153dc11a56,http://www.madrid.org/cs/Satellite?op2=PCON&idPagina=1204201624785&c=CM_ConvocaPrestac_FA&pagename=PortalContratacion%2FPage%2FPCON_contra

In [ ]:
display(resumen_status)